# Notebook 1 — Exploratory Data Analysis of FER2013

## Objective

This notebook analyses the FER2013 dataset before any preprocessing or training.
FER2013 contains 35,887 grayscale 48×48 face images labelled with seven discrete emotions: *Angry*, *Disgust*, *Fear*, *Happy*, *Sad*, *Surprise*, and *Neutral*.
Understanding its class distribution, visual variability, and inter-class similarity is essential to justify the preprocessing and augmentation choices made in Notebook 2.

**Business connection.** The sales-receptivity system must recognise these seven states in real time during a presentation.
Knowing which emotions are rare and which are visually confusable directly shapes class weighting, augmentation strategy, and model selection — all decisions documented here.

**What this notebook produces:**
- Quantified class imbalance across train / val / test splits
- Visual inspection of random samples per emotion
- Image quality characterisation (pixel variance, mean intensity)
- Mean-face heatmaps per class
- Pairwise cosine similarity between class prototypes
- A written summary of decisions carried forward to Notebook 2

## Section 1 — Setup and Data Loading

We fix random seeds for NumPy, Python's `random` module, and TensorFlow before any computation so that the validation split and random samples drawn later are reproducible across machines.
The seed value comes from `src.config.RANDOM_SEED` (42) so that every notebook in the project shares the same constant.
We also verify GPU availability here — not strictly needed for EDA, but confirming the environment at the top of every notebook is good practice and makes debugging easier.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Make src importable when running from the notebooks/ directory
sys.path.insert(0, str(Path("..").resolve()))

import tensorflow as tf
from src.config import RAW_DIR, EMOTION_LABELS, RANDOM_SEED
from src.data.loader import load_fer2013

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid")

print(f"TensorFlow {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

`load_fer2013` reads the Kaggle folder layout (`train/<emotion>/*.jpg`, `test/<emotion>/*.jpg`), then carves a stratified 15 % validation set from the Kaggle training split.
We use the Kaggle test split as-is — rather than re-splitting the full dataset — because this is the standard evaluation protocol for FER2013, enabling fair comparison with published results.
The stratified split ensures that each emotion's proportion is preserved in validation, which we can verify visually in Section 2.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_fer2013(RAW_DIR)

print(f"Train : {X_train.shape}  —  {len(X_train):,} images")
print(f"Val   : {X_val.shape}  —  {len(X_val):,} images")
print(f"Test  : {X_test.shape}  —  {len(X_test):,} images")
print(f"\nTotal : {len(X_train) + len(X_val) + len(X_test):,} images")
print(f"dtype : {X_train.dtype}  |  value range: [{X_train.min()}, {X_train.max()}]")

## Section 2 — Class Distribution

The table below shows absolute counts and class percentages for every emotion in each split.
Class imbalance in FER2013 is well-documented: *Happy* has roughly 13× as many training samples as *Disgust*, which is the most extreme case.
Knowing these proportions is the first step toward deciding how to counter imbalance — in this project we will use `compute_class_weight("balanced")` and targeted augmentation for minority classes.
Comparing percentages across splits also confirms that the stratified split preserved proportions correctly.

In [ ]:
splits = {"Train": y_train, "Val": y_val, "Test": y_test}
rows = []
for idx, emotion in enumerate(EMOTION_LABELS):
    row = {"Emotion": emotion.capitalize()}
    for split_name, labels in splits.items():
        count = int((labels == idx).sum())
        pct = count / len(labels) * 100
        row[split_name] = f"{count:,}  ({pct:.1f} %)"
    rows.append(row)

df_dist = pd.DataFrame(rows).set_index("Emotion")
df_dist

The bar chart below makes the magnitude of the imbalance immediately apparent — the *Happy* bar dwarfs *Disgust* and *Fear*.
Showing all three splits side by side confirms proportions are consistent, validating the stratified split.

In [ ]:
palette = sns.color_palette("Set2", len(EMOTION_LABELS))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (split_name, labels) in zip(axes, splits.items()):
    counts = [(labels == i).sum() for i in range(len(EMOTION_LABELS))]
    ax.bar(EMOTION_LABELS, counts, color=palette)
    ax.set_title(split_name, fontsize=12)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=35)

fig.suptitle("Class Distribution — FER2013", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 3 — Sample Grid per Class

Displaying raw images is the fastest sanity check: it confirms that the loader matched folder names to label indices correctly and exposes the intra-class visual variability that makes FER2013 challenging.
We draw eight images per class (7 × 8 grid) using the fixed seed so the selection is reproducible.
Pay attention to *Fear* vs. *Surprise* (both feature open mouths and wide eyes) and *Sad* vs. *Neutral* (distinguished mainly by brow tension) — these visual overlaps predict where the model will accumulate confusion-matrix errors.

In [ ]:
N_SAMPLES = 8
rng = np.random.default_rng(RANDOM_SEED)
fig, axes = plt.subplots(len(EMOTION_LABELS), N_SAMPLES, figsize=(14, 11))

for row_idx, emotion in enumerate(EMOTION_LABELS):
    class_idx = np.where(y_train == row_idx)[0]
    sample_idx = rng.choice(class_idx, size=N_SAMPLES, replace=False)
    for col_idx, img_idx in enumerate(sample_idx):
        ax = axes[row_idx, col_idx]
        ax.imshow(X_train[img_idx], cmap="gray")
        ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(
                emotion.capitalize(), fontsize=9, rotation=0,
                labelpad=50, va="center"
            )

fig.suptitle("8 Random Samples per Emotion Class — Training Set", fontsize=13)
plt.tight_layout()
plt.show()

## Section 4 — Image Quality Analysis

Pixel variance measures how much intensity varies within a single image — a near-zero value indicates an almost-uniform frame (blank, completely dark, or washed-out) that carries little discriminative signal.
We histogram the per-image variance across the entire training set to see whether problematic images are isolated outliers or a systemic quality issue.
Decision: if fewer than ~1 % of images fall below a reasonable variance threshold, we keep them; FER2013 is an established benchmark and removing samples could undermine comparisons with published results.

In [ ]:
variances = X_train.reshape(len(X_train), -1).var(axis=1).astype(np.float32)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(variances, bins=100, color=palette[0], edgecolor="none")
ax.set_xlabel("Per-image pixel variance")
ax.set_ylabel("Image count")
ax.set_title("Distribution of Pixel Variance — Training Set")
plt.tight_layout()
plt.show()

low_var = variances < 100
print(f"Images with variance < 100: {low_var.sum():,}  ({low_var.mean()*100:.2f} % of train set)")

Mean pixel intensity flags over- and underexposed images: values near 0 indicate completely dark frames, values near 255 indicate completely white frames.
Neither extreme provides usable facial texture, so knowing their proportion tells us whether an explicit cleaning step is justified.

In [ ]:
means = X_train.reshape(len(X_train), -1).mean(axis=1).astype(np.float32)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(means, bins=100, color=palette[1], edgecolor="none")
ax.set_xlabel("Per-image mean pixel intensity (0 – 255)")
ax.set_ylabel("Image count")
ax.set_title("Distribution of Mean Pixel Intensity — Training Set")
plt.tight_layout()
plt.show()

print(f"Mean < 20   (very dark)  : {(means < 20).sum():,}")
print(f"Mean > 220  (very bright): {(means > 220).sum():,}")

Inspecting the five lowest-variance images visually distinguishes genuinely corrupted frames from dim-but-valid faces.
A corrupted image (e.g. solid colour block) might warrant removal, while a dim face with recognisable features is fine — pixel normalisation during preprocessing will bring its intensity into a usable range.

In [ ]:
lowest_idx = np.argsort(variances)[:5]

fig, axes = plt.subplots(1, 5, figsize=(11, 2.8))
for ax, img_idx in zip(axes, lowest_idx):
    emotion_name = EMOTION_LABELS[y_train[img_idx]]
    ax.imshow(X_train[img_idx], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"var = {variances[img_idx]:.1f}\n{emotion_name}", fontsize=8)
    ax.axis("off")

fig.suptitle("5 Lowest-Variance Images — Training Set", fontsize=11)
plt.tight_layout()
plt.show()

## Section 5 — Mean Intensity Heatmap per Class

The mean face for each class is the pixel-wise average over all training images of that class, rendered as a heatmap.
It blurs away individual features but reveals where discriminative signal concentrates on average — the mouth region for *Happy*, the brow for *Angry*, or the wide-open eyes for *Surprise*.
These spatial priors are consistent with what Grad-CAM should highlight in Notebook 4; seeing them here provides an early consistency check before any model is trained.

In [ ]:
fig, axes = plt.subplots(1, len(EMOTION_LABELS), figsize=(14, 3))

for idx, emotion in enumerate(EMOTION_LABELS):
    class_imgs = X_train[y_train == idx].astype(np.float32)
    mean_img = class_imgs.mean(axis=0)
    ax = axes[idx]
    im = ax.imshow(mean_img, cmap="hot", vmin=0, vmax=255)
    ax.set_title(emotion.capitalize(), fontsize=10)
    ax.axis("off")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, label="Mean intensity")
fig.suptitle("Mean Face per Emotion Class — Training Set", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

## Section 6 — Visual Similarity between Classes

We compute the pairwise cosine similarity between the flattened mean faces for all seven classes.
A value near 1.0 means the two mean faces are nearly co-linear in pixel space, indicating that the model will find the pair harder to separate; lower values correspond to more distinct prototypes.
This quantifies the intuition from Section 3: *Fear*/*Surprise* and *Sad*/*Neutral* should appear as the closest pairs, predicting the largest off-diagonal entries in the eventual confusion matrix.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

mean_faces = np.array([
    X_train[y_train == idx].astype(np.float32).mean(axis=0).flatten()
    for idx in range(len(EMOTION_LABELS))
])
sim_matrix = cosine_similarity(mean_faces)

labels_cap = [e.capitalize() for e in EMOTION_LABELS]
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt=".4f",
    cmap="YlOrRd",
    xticklabels=labels_cap,
    yticklabels=labels_cap,
    vmin=0.95,
    vmax=1.0,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Pairwise Cosine Similarity of Class Mean Faces", fontsize=12)
plt.tight_layout()
plt.show()

## Section 7 — Conclusions and Decisions for Preprocessing

The EDA yields the following actionable decisions, carried forward to Notebook 2:

| Finding | Decision |
|---|---|
| *Happy* has ~13× more samples than *Disgust* | Use `compute_class_weight("balanced")` from scikit-learn during training |
| *Disgust* (~600 train samples) and *Fear* are strongly under-represented | Apply heavier augmentation to these two classes in the training generator |
| Low-variance images are < 0.5 % of the dataset and appear dim-but-valid | No cleaning pipeline needed; pixel normalisation in preprocessing is sufficient |
| *Fear* ↔ *Surprise* and *Sad* ↔ *Neutral* have the highest cosine similarity | Expect these pairs to dominate confusion-matrix errors; monitor them in Notebook 4 |
| *Angry* ↔ *Disgust* are also visually close | Same expectation; the receptivity mapper assigns equal scores to both, which is consistent |
| Native FER2013 resolution is 48 × 48 | Keep 48 × 48 for the custom CNN; resize to 64 × 64 for MobileNetV2 (minimum viable input) |
| Images are uint8 grayscale | Normalise to [0, 1] float32 during preprocessing; stack to 3-channel for MobileNetV2 |

## Section 8 — Summary and Link to the Next Notebook

This notebook characterised FER2013 from three complementary angles: statistical (class counts and proportions), visual (sample grid and mean-face heatmaps), and structural (pixel variance, mean intensity, cosine similarity).
The central takeaway is that the dataset is clean enough to use as-is — no images need to be removed — but the class imbalance is severe and must be countered explicitly during training.
The hardest pairs (*Fear*/*Surprise* and *Sad*/*Neutral*) are already identified, which will make the confusion-matrix analysis in Notebook 4 easier to interpret in terms of business impact.

**Next step → Notebook 2 (`02_preprocessing.ipynb`):** apply pixel normalisation, build the `ImageDataGenerator` pipelines with targeted augmentation for *Disgust* and *Fear*, compute class weights with scikit-learn, and save processed arrays to `data/processed/`.